<a href="https://colab.research.google.com/github/jetisaplane1/AI-for-Business-BUS4-118S/blob/dev/Exercise1_PromptChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import json
import re

def mock_call_llm(instructions: str, user_input: str) -> str:
    """
    Mock LLM: returns deterministic outputs based on the user's message + instructions.
    This is for practicing prompt engineering without an API.
    """

    # STEP 1: classifier (JSON-only)
    if "triage assistant" in instructions.lower() and "output only valid json" in instructions.lower():
        msg = user_input.lower()

        category = "other"
        confidence = 0.62
        entities = {"order_id": None, "email": None, "product": None, "error_message": None}
        reason = "Insufficient information."

        # extract email if present
        email_match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', user_input)
        if email_match:
            entities["email"] = email_match.group(0)

        # classify login issues
        if "log in" in msg or "login" in msg or "invalid session" in msg:
            category = "login"
            confidence = 0.91
            reason = "User reports login failure with session error."
            if "invalid session" in msg:
                entities["error_message"] = "invalid session"

        return json.dumps({
            "category": category,
            "confidence": confidence,
            "entities": entities,
            "reason": reason
        }, indent=2)

    # STEP 2: missing info questions
    if "ask at most 3 questions" in instructions.lower():
        # Look for category in embedded JSON
        if "login" in user_input.lower():
            return "\n".join([
                "1. What browser and operating system are you using?",
                "2. Does the issue happen in an incognito/private window as well?",
                "3. Are you able to log in successfully from another device or network?"
            ])
        return "\n".join([
            "1. Can you share any error message you see?",
            "2. When did the issue start?",
            "3. What device/app are you using?"
        ])

    # STEP 3: solution
    if "step-by-step fix" in instructions.lower() or "provide:" in instructions.lower():
        # If they can log in on phone but not desktop, suggest browser/session steps
        return (
            "## Diagnosis\n"
            "This looks like a browser-specific session or cached authentication issue on your desktop.\n\n"
            "## Steps to Try\n"
            "1. Open an **Incognito/Private** window and try logging in.\n"
            "2. If that works, clear **cookies + cache** for the site and restart the browser.\n"
            "3. Disable extensions temporarily (especially ad blockers/password managers) and retry.\n"
            "4. Update Chrome to the latest version.\n"
            "5. Restart your computer and try again.\n\n"
            "## If It Still Fails\n"
            "Reply with the exact error message and approximate time it occurs, and we can reset your session server-side."
        )

    # STEP 4: escalation decision (JSON-only)
    if "escalate" in instructions.lower() and "output only json" in instructions.lower():
        # Default: do not escalate if confidence high and steps available
        return json.dumps({
            "escalate": False,
            "reason": "Common login/session issue; troubleshooting steps provided and no fraud indicators.",
            "handoff_summary": (
                "Customer reports 'invalid session' error when logging in on desktop. "
                "Email provided. Issue appears browser-specific; login works on phone. "
                "Provided incognito, cache/cookies, extension disable, update/restart steps."
            )
        }, indent=2)

    # fallback
    return "MockLLM: No matching rule for this prompt."

def call_llm(instructions: str, user_input: str) -> str:
    return mock_call_llm(instructions, user_input)

# ----------------------------
# Prompt chain (same structure)
# ----------------------------

customer_message = "I can’t log in since yesterday. It says 'invalid session'. My email is jet@example.com."

# STEP 1 — classify (JSON only)
instr1 = (
    "You are a customer support triage assistant. Output ONLY valid JSON (no code fences, no extra text). "
    "Classify the issue into one of: billing, login, shipping, refund, product_defect, other. "
    "Extract entities if present: order_id, email, product, error_message. "
    "Return confidence 0–1 and a short reason (max 20 words)."
)

out1_raw = call_llm(instr1, f"Customer message: {customer_message}")
print("STEP 1 RAW:\n", out1_raw)

step1 = json.loads(out1_raw)
print("\nSTEP 1 PARSED JSON:\n", step1)

# STEP 2 — ask missing info questions
instr2 = (
    "You are a helpful support agent. Use the Step1 JSON. "
    "Ask at most 3 questions, only for missing info required to solve the category. "
    "Tone: calm, concise, professional. Output as a numbered list."
)

out2 = call_llm(instr2, f"Step1 JSON: {json.dumps(step1)}")
print("\nSTEP 2 QUESTIONS:\n", out2)

# Simulate customer answers
customer_answers = "I’m using Chrome on Windows. Clearing cookies didn’t help. I can log in on my phone though."

# STEP 3 — solution
instr3 = (
    "Use Step 1 category + customer answers. Provide: "
    "(1) short diagnosis, (2) step-by-step fix (max 6 steps), (3) what to do if it fails. "
    "Do NOT request passwords, SSNs, or full card numbers. Output in Markdown with headings."
)

out3 = call_llm(instr3, f"Step1 JSON: {json.dumps(step1)}\nCustomer answers: {customer_answers}")
print("\nSTEP 3 SOLUTION:\n", out3)

# STEP 4 — escalation
instr4 = (
    "Decide whether to escalate to a human. Escalate if: confidence < 0.6 OR missing critical info "
    "after questions OR user is angry/threatening chargeback OR potential fraud. "
    "Output ONLY JSON with keys: escalate (bool), reason (string), handoff_summary (string, max 80 words)."
)

out4_raw = call_llm(instr4, f"Step1 JSON: {json.dumps(step1)}\nCustomer answers: {customer_answers}\nProposed solution: {out3}")
print("\nSTEP 4 RAW:\n", out4_raw)

step4 = json.loads(out4_raw)
print("\nSTEP 4 PARSED JSON:\n", step4)

STEP 1 RAW:
 {
  "category": "login",
  "confidence": 0.91,
  "entities": {
    "order_id": null,
    "email": "jet@example.com",
    "product": null,
    "error_message": "invalid session"
  },
  "reason": "User reports login failure with session error."
}

STEP 1 PARSED JSON:
 {'category': 'login', 'confidence': 0.91, 'entities': {'order_id': None, 'email': 'jet@example.com', 'product': None, 'error_message': 'invalid session'}, 'reason': 'User reports login failure with session error.'}

STEP 2 QUESTIONS:
 1. What browser and operating system are you using?
2. Does the issue happen in an incognito/private window as well?
3. Are you able to log in successfully from another device or network?

STEP 3 SOLUTION:
 ## Diagnosis
This looks like a browser-specific session or cached authentication issue on your desktop.

## Steps to Try
1. Open an **Incognito/Private** window and try logging in.
2. If that works, clear **cookies + cache** for the site and restart the browser.
3. Disable exte